# 02 — Metric selection

Why metric choice matters: compare MAE, RMSE, and MAPE on the same forecasts. MAPE can mislead when actuals are near zero (e.g. holidays or low-volume series).

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
from src.evaluation import train_val_split, mae, rmse, mape
data_path = root / "data" / "surgeries.csv"
if not data_path.exists():
    from src.data_generator import generate_surgery_counts
    generate_surgery_counts(output_path=str(data_path))
df = pd.read_csv(data_path, parse_dates=["date"])

## Two series: high-volume vs low-volume

We use a simple baseline (last known value, i.e. naive forecast) and compare metrics.

In [ ]:
# High-volume: General Surgery @ City General (all hospitals sum)
high = df.groupby("date")["surgery_count"].sum().reset_index()
high = high.sort_values("date").reset_index(drop=True)
train_high, val_high = train_val_split(high, date_col="date", val_days_or_ratio=0.2)
last_train = train_high["surgery_count"].iloc[-1]
pred_high = np.full(len(val_high), last_train)
print("High-volume (daily total):")
print(f"  MAE:  {mae(val_high['surgery_count'], pred_high):.2f}")
print(f"  RMSE: {rmse(val_high['surgery_count'], pred_high):.2f}")
print(f"  MAPE (skip zeros): {mape(val_high['surgery_count'], pred_high, skip_zeros=True):.1f}%")
try:
    print(f"  MAPE (include zeros): {mape(val_high['surgery_count'], pred_high, skip_zeros=False):.1f}%")
except Exception as e:
    print(f"  MAPE (include zeros): {e}")

In [ ]:
# Low-volume: one specialty at one hospital (can have zeros on holidays)
low = df[(df["specialty_name"] == "Ophthalmology") & (df["hospital_id"] == "H3")][["date", "surgery_count"]].copy()
low = low.sort_values("date").reset_index(drop=True)
train_low, val_low = train_val_split(low, date_col="date", val_days_or_ratio=0.2)
last_train_low = train_low["surgery_count"].iloc[-1]
pred_low = np.full(len(val_low), last_train_low)
print("Low-volume (Ophthalmology @ H3):")
print(f"  MAE:  {mae(val_low['surgery_count'], pred_low):.2f}")
print(f"  RMSE: {rmse(val_low['surgery_count'], pred_low):.2f}")
print(f"  MAPE (skip zeros): {mape(val_low['surgery_count'], pred_low, skip_zeros=True):.1f}%")
try:
    m = mape(val_low["surgery_count"], pred_low, skip_zeros=False)
    print(f"  MAPE (include zeros): {m:.1f}%")
except Exception as e:
    print(f"  MAPE (include zeros): undefined or huge — {e}")

## Takeaway

- **MAE** and **RMSE** are in the same units as the target; RMSE penalizes large errors more.
- **MAPE** is scale-free but explodes or is undefined when actuals are zero (or very small). For surgery counts with holidays/weekends, use `skip_zeros=True` or prefer MAE/RMSE for low-volume series.